In [29]:
from typing import Annotated, TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from src.retrieval.embedders import SentenceTransformerEmbedder
from src.retrieval.store import ElasticSearchVectorStore

embedder = SentenceTransformerEmbedder()
cve_store = ElasticSearchVectorStore(index_name='cve_index')
mitre_store = ElasticSearchVectorStore(index_name='mitre_attack')

@tool
def query_cve(query: str) -> str:
    '''Tool to query CVEs from the vector store. Useful for finding vulnerabilities related to specific software, attack techniques, or CVE IDs.'''
    vector = embedder.embed_query(query)
    results = cve_store.search(vector, top_k=3)
    return '\n\n'.join([f"{res['id']}: {res['description']}" for res in results])

@tool
def query_mitre(query: str) -> str:
    '''Tool to query MITRE techniques from the vector store. Useful for finding information about specific attack techniques.'''
    vector = embedder.embed_query(query)
    results = mitre_store.search(vector, top_k=3)
    return '\n\n'.join([f"{res['id']}: {res['description']}" for res in results])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9290.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
class AgentState(TypedDict):
    messages: Annotated[BaseMessage, add_messages]

llm = ChatOllama(model='llama3.1:8b')

available_tools = [query_cve, query_mitre]
llm_with_tools = llm.bind_tools(available_tools)

SYSTEM_PROMPT = '''
You are a SOC expert analyst. Your work is to analyze security alerts, logs and threat intelligence to identify potential security incidents and recommend appropriate responses.
Your goal is to analyze security alerts and enrich them with relevant information from CVE and MITRE databases.
YOU HAVE TOOLS AVAILABLE TO YOU, USE THEM WHEN NECESSARY TO COMPLETE YOUR TASKS.

GOLD RULE:
If user provides you with an alert, you MUST analyze it and enrich it with relevant information from CVE and MITRE databases. 
You MUST use the tools available to you to retrieve this information. You MUST NOT provide a final answer without using the tools to retrieve relevant information.

ROUTING RULES:
1. Behaviors, commands, processes or malware -> USE query_mitre TOOL
2. Software versions, scan results or vulnerability identifiers -> USE query_cve TOOL
3. Vulnerabilities + Post-explotation behaviors -> USE BOTH TOOLS
4. If you need to retrieve information to complete your tasks, ALWAYS USE THE TOOLS.
5. If do not know which tool to use, USE BOTH TOOLS.

FINAL ANSWER RULES:
ONLY when you have retrieved relevant information from the tools, you can provide a final answer.
Use EXACTLY this structure in Markdown format for your final answer:
### Incident Summary
[Summary]
### Intelligence Correlation
[CVEs and MITRE explanations. If both, explain relationship between them]
### Priority and Severity
[Your assessment of the priority and severity of the incident, based on the information you have]
### Recommended Response
[Your recommended response to the incident, including any immediate actions that should be taken and any further investigation that may be needed]
Answer in a structured, professional and concise way.
'''

def agent_node(state: AgentState):
    '''Agent node that analyze the alert and decides if use tools or writes a final answer.'''
    messages = state['messages']
    
    if not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages

    response = llm_with_tools.invoke(messages)
    print(f'DEBUG: Tool calls: {response.tool_calls}')
    return {'messages': [response]}


def use_tool_decision(state: AgentState):
    '''Decision node that determines whether the agent should use a tool or not.'''
    last_message = state['messages'][-1]
    
    if hasattr(last_message, 'tool_calls') and len(last_message.tool_calls) > 0:
        return 'action'
    
    return 'end'

In [10]:
workflow = StateGraph(AgentState)
workflow.add_node('agent', agent_node)
workflow.add_node('action', ToolNode(available_tools))
workflow.set_entry_point('agent')
workflow.add_conditional_edges(
    'agent',
    use_tool_decision,
    {
        'action': 'action',
        'end': END
    }
)

workflow.add_edge('action', 'agent')
app = workflow.compile()

In [3]:
user_input = input("\nUser: ")

inputs = {'messages': [HumanMessage(content=user_input)]}
for event in app.stream(inputs, stream_mode='values'):
    last_message = event['messages'][-1]
    if hasattr(last_message, 'content') and last_message.content and not hasattr(last_message, 'tool_calls'):
        print(f"Agent: {last_message.content}")

Agent: hola
Agent: CVE-2025-59832: Horilla is a free and open source Human Resource Management System (HRMS). Prior to version 1.4.0, there is a stored XSS vulnerability in the ticket comment editor. A low-privilege authenticated user could run arbitrary JavaScript in an admin’s browser, exfiltrate the admin’s cookies/CSRF token, and hijack their session. This issue has been patched in version 1.4.0.

CVE-2026-40867: Horilla is a free and open source Human Resource Management System (HRMS). In 1.5.0, a broken access control vulnerability in the helpdesk attachment viewer allows any authenticated user to view attachments from other tickets by changing the attachment ID. This can expose sensitive support files and internal documents across unrelated users or teams.

CVE-2025-48869: Horilla is a free and open source Human Resource Management System (HRMS). Unauthenticated users can access uploaded resume files in Horilla 1.3.0 by directly guessing or predicting file URLs. These files are 

In [ ]:
# Invoke
user_input = 'EDR alert: Detected CVE-2021-44228 exploitation attempt on web server. '

inputs = {'messages': [HumanMessage(content=user_input)]}
messages = app.invoke(inputs)
for m in messages["messages"]:
    m.pretty_print()

DEBUG: Tool calls: []
================================ Human Message =================================

EDR alert: Detected CVE-2021-44228 exploitation attempt on web server. 
================================== Ai Message ==================================

### Incident Summary
EDR alert indicates a possible exploitation attempt of CVE-2021-44228 on a web server.

### Intelligence Correlation
We will use the `query_cve` tool to gather information about the identified vulnerability.
The MITRE database may also provide additional context or related attack techniques that an attacker could utilize.

### Priority and Severity
**Priority: High**
**Severity: Critical**

### Recommended Response
Firstly, we recommend investigating the web server's logs for any suspicious activity related to this exploitation attempt. 
Next, ensure that all software on the affected system is up-to-date, especially Log4j version 2.
Further investigation should be conducted to identify potential entry points and

In [11]:
for event in app.stream(inputs, stream_mode='updates'):
    for node_name, node_state in event.items():
        print(f'\nNODO: [{node_name}]')

        last_message = node_state['messages'][-1]

        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            for tool_call in last_message.tool_calls:
                print(f"Tool call: {tool_call}")

        elif getattr(last_message, 'type', '') == 'tool':
            resp = last_message.content[:150].replace('\n', ' ')
            print(f"Tool response [{last_message.name}]: {resp}...")

        elif getattr(last_message, 'type', '') == 'ai':
            print(f"AI: {last_message.content}")
            


DEBUG: Tool calls: []

NODO: [agent]
AI: ### Incident Summary
EDR alert detected CVE-2021-44228 exploitation attempt on a web server.

### Intelligence Correlation
The detected CVE is related to a known vulnerability in Apache Log4j, tracked as CVE-2021-44228 by the Common Vulnerabilities and Exposures (CVE) database. This vulnerability allows an attacker to execute arbitrary code on the affected system through a crafted log message. According to MITRE, this vulnerability is associated with the attack technique T1190 - Exploit Use of Out-of-Date Software.

### Priority and Severity
High priority, High severity. The exploitation attempt indicates potential compromise of the web server, which could lead to data theft or other malicious activities.

### Recommended Response
Immediately investigate the web server for signs of compromise and take remediation steps to patch the vulnerability. This may involve applying a security update or reconfiguring the Log4j version to a secure one. Moni

## Prueba sin tools

In [30]:
import os
from typing import Annotated, TypedDict, List
from pydantic import BaseModel, Field
import json

from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langchain_ollama import ChatOllama


from src.retrieval.embedders import SentenceTransformerEmbedder, LocalvLLMEmbedder
from src.retrieval.store import ElasticSearchVectorStore

local_embedder = LocalvLLMEmbedder(model_name='bge-m3')
embedder = SentenceTransformerEmbedder()
emb = SentenceTransformerEmbedder('BAAI/bge-small-en-v1.5')
cve_store = ElasticSearchVectorStore(index_name='cve_index')
mitre_store = ElasticSearchVectorStore(index_name='mitre_attack')
mitre_store_v2 = ElasticSearchVectorStore(index_name='mitre_attack_v2')
mitre_store_bge = ElasticSearchVectorStore(index_name='mitre_attack_bge')
mitre_store_v2_bge = ElasticSearchVectorStore(index_name='mitre_attack_v2_bge_small')
# llm = ChatOllama(model='llama3.1:8b')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7797.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10142.99it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
from langchain_google_genai import ChatGoogleGenerativeAI
from src.agent.token_tracker import UniversalTokenTracker
from src.agent.config import AgentConfig
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv('GEMINI_TOKEN')

llm = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite', api_key=API_KEY, temperature=0.2)

# llm = ChatOpenAI(
#     model="gpt-oss-20b",
#     api_key='EMPTY',
#     temperature=0.2,
#     base_url="http://10.0.152.198:8001/v1"
# )

# llm = ChatOpenAI(
#     model="gemma-4-26b-a4b",
#     api_key='EMPTY',
#     # streaming=True,
#     # stream_usage=True,
#     temperature=0.2,
#     # max_tokens=None,
#     # timeout=12000,
#     # reasoning_effort="low",
#     # max_retries=3,
#     base_url="http://10.0.152.198:8003/v1",
#     # http_client=httpx.Client(timeout=httpx.Timeout(connect=60.0, read=600.0, write=60.0, pool=60.0))
# )

tracker = UniversalTokenTracker()
tracker.reset()
config = AgentConfig(
        use_context_window=False,
        context_window_size=10,
        generate_report=False,
        report_dir='reports',
        mitre_top_k=10
    )


In [32]:
import os
from langchain_core.messages import SystemMessage, HumanMessage
from src.models.models import AlertClasification, AgentState, ValidationReport
import time


def classification_node(state: AgentState) -> AgentState:
    '''Node that classifies the alert and extracts relevant keywords for MITRE and CVE searches.'''
    
    tracker.set_current_node('classification')  
    start_time = time.time()

    prompt = f'''
    You are a SOC analyst. Your task is to analyze the following security alert and determine if it contains information that can be used to search in MITRE and CVE databases.

    Alert: {state['original_alert']}

    For MITRE, you are looking for behaviors, commands, processes, tactics or malware mentioned in the alert. If you find any, set mitre_search to True and extract the relevant keywords for searching in MITRE database.

    For CVE, you are looking for software versions, scan results or vulnerability identifiers mentioned in the alert. If you find any, set cve_search to True and extract the relevant keywords for searching in CVE database.
    
    Also, create a technical description of the alert that can be used as a query for the searches in MITRE and CVE databases. DO NOT make assumptions, just explain the alert with details.
    
    EXAMPLES:
    - Alert: "Multiple failed login attempts in RDP" mitre_search: True (brute force), cve_search: False (no vulnerabilities mentioned)
    - Alert: "WINRAR old version detected that allows remote code execution" mitre_search: False, cve_search: True (WINRAR vulnerability)
    - Alert: "Mimikatz process detected running in memory" mitre_search: True (credential dumping), cve_search: False (no vulnerabilities mentioned)
    - Alert: "Proxyshell exploitation (CVE-2021-34473) followed by webshell detected" mitre_search: True (Web Shell), cve_search: True (Proxyshell vulnerability)
    
    At the slightest suspicion of malicious behavior, you MUST set mitre_search to True, and if there is any mention of exploitable software, vulnerabilities or patches, you MUST set cve_search to True.
    '''
    structured_llm = llm.with_structured_output(AlertClasification)
    tracked_llm = structured_llm.with_config(callbacks=[tracker])
    classification = tracked_llm.invoke([SystemMessage(content=prompt),
                                            HumanMessage(content=f'The alert to analyze is: {state["original_alert"]}')])
    
    tracker.record_node_time('classification', time.time() - start_time)
    return {'classification': classification}



def mitre_search_node(state: AgentState) -> AgentState:
    '''Node that performs MITRE search if mitre_search is True.'''

    if not state['classification'].mitre_search:
        return {'mitre_data': 'No relevant MITRE information found in the alert.'}
    
    tracker.set_current_node(node_name='mitre_search_node')
    start_time = time.time()

    query = state['classification'].mitre_description
    query_vector = embedder.embed_query(query)
    results = mitre_store.search(query_vector, top_k=config.mitre_top_k)

    results_text = 'MITRE results:\n' + '\n'.join([f"Technique ID: {r['technique_id']}\n"
                    f"Name: {r['name']}\n"
                    f"Description: {r['description']}\n"
                    f"Tactics: {r['tactics']}\n"
                    f"Platforms: {r['platforms']}\n"
                    "-----------------" for r in results])
    
    tracker.record_node_time('mitre_search_node', time.time() - start_time)
    return {'mitre_data': results_text}

def cve_search_node(state: AgentState) -> AgentState:
    '''Node that performs CVE search if cve_search is True.'''

    if not state['classification'].cve_search:
        return {'cve_data': 'No relevant CVE information found in the alert.'}

    tracker.set_current_node('cve_search_node')
    start_time = time.time()
    
    query = state['classification'].cve_description
    query_vector = embedder.embed_query(query)
    results = cve_store.search(query_vector, top_k=3)

    results_text = 'CVE results:\n' + '\n'.join([f"CVE ID: {r['id']}\n"
                    f"Title: {r['title']}\n"
                    f"Description: {r['description']}\n"
                    f"Published Date: {r['published_date']}\n"
                    f"CVSS: {r['cvss']}\n"
                    f"Affected Products: {r['versions']}\n"
                    f"Mitigations: {r['mitigations']}\n"
                    f"SSVC: {r['ssvc']}\n"
                    f"References: {r['references']}\n"
                    f"In KEV: {r['in_kev']}\n"
                    "-----------------" for r in results])
    
    tracker.record_node_time('cve_search_node', time.time() - start_time)
    return {'cve_data': results_text}

def alert_context_node(state: AgentState) -> AgentState:
    '''Node that retrieves previous and subsequent alerts to provide context to the agent.'''
    if not config.use_context_window:
        return {'context_window': 'Context window is disabled by configuration.'}
    
    try:
        tracker.set_current_node('alert_context_node')
        start_time = time.time()

        alert = state['original_alert']
        context_window = alert_data.get_context_window(alert, config.context_window_size)
        
        tracker.record_node_time('alert_context_node', time.time() - start_time)
    
    except Exception as e:
        print(f"Error occurred while retrieving alert context: {e}")
        context_window = []
    return {'context_window': context_window}

def validation_node(state: AgentState) -> AgentState:
    '''Node that validates the relevance of the retrieved MITRE and CVE information.'''
    
    tracker.set_current_node('validation_node')
    start_time = time.time()

    prompt = f'''
    You are a SOC analyst. Your task is to evaluate the relevance of the retrieved MITRE techniques and CVE vulnerabilities from an automatic search system.
    You must evaluate if each result is actually relevant to the original alert or whether it is semantic noise (for example, a MITRE technique that shares some keywords with the alert but is not actually related to the attack described in the alert).
    You also have a context window with previous and subsequent alerts that can help you understand better the situation and the relevance of the retrieved information. Use it to determine if the retrieved MITRE techniques and CVE vulnerabilities are actually relevant or are noise.
    
    ORIGINAL ALERT: {state['original_alert']}

    CONTEXT WINDOW: {state['context_window']}

    EXTRACTED MITRE RESULTS: {state['mitre_data']}
    
    EXTRACTED CVE RESULTS: {state['cve_data']}

    INSTRUCTIONS:
    - Compare each technique and vulnerability with the original alert.
    - For each MITRE technique retrieved, assign a relevance score from 0 to 1, where 0 means completely irrelevant and 1 means highly relevant. 
    - Provide a detailed explanation of why you assigned that score.
    - Make a final decision on whether this technique should be included in the final report or not (decision=True/False).
    '''

    validator = llm.with_structured_output(ValidationReport)
    validation_report = validator.invoke([HumanMessage(content=prompt)], callbacks=[tracker])

    tracker.record_node_time('validation_node', time.time() - start_time)

    for evaluation in validation_report.mitre_evaluations + validation_report.cve_evaluations:
        print(f"ID {evaluation.item_id} - Relevance Score: {evaluation.relevance_score}, Decision: {evaluation.decision}\nExplanation: {evaluation.explanation}\n")
    return {'validation_report': validation_report}


def final_report_node(state: AgentState) -> AgentState:
    '''Node that generates a final report based on the original alert, the classification, and the retrieved MITRE and CVE information.'''
    if not config.generate_report:
        return {'final_report': 'Report generation is disabled by configuration.'}
    
    tracker.set_current_node('final_report_node')
    start_time = time.time()

    validated_data = state['validation_report']
    validated_mitre = [e for e in validated_data.mitre_evaluations if e.decision]
    validated_cve = [e for e in validated_data.cve_evaluations if e.decision]
    
    prompt = f'''
    You are a SOC analyst. Your task is to generate a final report based on the information about the alert, the relevant MITRE techniques and its tactics and CVE vulnerabilities that have been identified.
    Generate a concise report in Markdown format summarizing the incident, correlating the MITRE and CVE information with the original alert, and providing an assessment of the priority and severity of the incident, as well as recommended response actions.
    '''

    user_input = f'''
    Original Alert: {state['original_alert']}
    Classification: {state['classification']}
    CONTEXT WINDOW: {state['context_window']}
    MITRE Information: {validated_mitre}
    CVE Information: {validated_cve}'''

    response = llm.invoke([SystemMessage(content=prompt), HumanMessage(content=user_input)], callbacks=[tracker])
    alert_id = state['original_alert'].get('id', 'unknown_alert_id')
    # Escribir en un archivo de texto el informe final
    report_path = os.path.join(config.report_dir, f'final_report_{alert_id}.md')
    os.makedirs(config.report_dir, exist_ok=True)
    with open(report_path, 'w') as f:
        f.write(response.content[0]['text'])

    tracker.record_node_time('final_report_node', time.time() - start_time)
    
    return {'final_report': response.content[0]}


In [13]:
workflow = StateGraph(AgentState)
workflow.add_node('classification', classification_node)
workflow.add_node('mitre_search_node', mitre_search_node)
workflow.add_node('cve_search_node', cve_search_node)
workflow.add_node('alert_context_node', alert_context_node)
workflow.add_node('validation_node', validation_node)
workflow.add_node('final_report_node', final_report_node)

workflow.set_entry_point('classification')

workflow.add_edge('classification', 'mitre_search_node')
workflow.add_edge('classification', 'cve_search_node')
workflow.add_edge('classification', 'alert_context_node')

workflow.add_edge(['mitre_search_node', 'cve_search_node', 'alert_context_node'], 'validation_node')
workflow.add_edge('validation_node', 'final_report_node')

workflow.add_edge('final_report_node', END)

app = workflow.compile()

In [152]:
from typing import Annotated, TypedDict, List, Dict

class AgentState(TypedDict):
    original_alert: Dict
    context_window: List[str] = Field(description="A list of previous and post alerts that can provide context to the agent when analyzing the current alert. This can include previous alerts from the same source, alerts with similar characteristics, or any other relevant information that can help the agent understand the situation better.")
    
    classification: AlertClasification
    mitre_data1: Annotated[List[Dict], Field(description="Relevant MITRE information retrieved")]
    mitre_data2: Annotated[List[Dict], Field(description="Relevant MITRE information retrieved")]
    mitre_data3: Annotated[List[Dict], Field(description="Relevant MITRE information retrieved")]

    cve_data: Annotated[str, Field(description="Relevant CVE information retrieved")]

    validation_report: ValidationReport = Field(description="The validation report containing evaluations for MITRE and CVE items")

    final_report: Annotated[str, Field(description="Final report generated by the agent")]

def mitre_search_node(state: AgentState) -> AgentState:
    '''Node that performs MITRE search if mitre_search is True.'''

    if not state['classification'].mitre_search:
        return {'mitre_data': 'No relevant MITRE information found in the alert.'}
    
    # tracker.set_current_node(node_name='mitre_search_node')
    # start_time = time.time()

    query1 = state['classification'].mitre_description
    query2 = state['classification'].mitre_keywords
    query3 = state['original_alert']['rule']['description']

    prefix = "Represent this sentence for searching relevant passages: "

    query_vector1 = emb.embed_query(prefix + query1)
    query_vector2 = emb.embed_query(prefix + query2)
    query_vector3 = emb.embed_query(prefix + query3)

    results1 = mitre_store_v3_bge.search(query_vector1, top_k=config.mitre_top_k)
    results2 = mitre_store_v3_bge.search(query_vector2, top_k=config.mitre_top_k)
    results3 = mitre_store_v3_bge.search(query_vector3, top_k=config.mitre_top_k)

    # results1 = mitre_store_v2.search_mitre_hybrid(query_vector=query_vector1, query_text=query2, top_k=config.mitre_top_k)
    # results2 = mitre_store_v2.search_mitre_hybrid(query_vector=query_vector2, query_text=query2, top_k=config.mitre_top_k)
    # results3 = mitre_store_v2.search_mitre_hybrid(query_vector=query_vector3, query_text=query2, top_k=config.mitre_top_k)

    # results_text = 'MITRE results:\n' + '\n'.join([f"Technique ID: {r['technique_id']}\n"
    #                 f"Name: {r['name']}\n"
    #                 f"Description: {r['description']}\n"
    #                 f"Tactics: {r['tactics']}\n"
    #                 f"Platforms: {r['platforms']}\n"
    #                 "-----------------" for r in results])
    
    # tracker.record_node_time('mitre_search_node', time.time() - start_time)
    return {'mitre_data1': results1,
            'mitre_data2': results2,
            'mitre_data3': results3}

In [153]:
workflow = StateGraph(AgentState)
workflow.add_node('classification', classification_node)
workflow.add_node('mitre_search_node', mitre_search_node)

workflow.set_entry_point('classification')
workflow.add_edge('classification', 'mitre_search_node')
workflow.add_edge('mitre_search_node', END)
# workflow.add_edge('classification', END)
app = workflow.compile()

In [ ]:
user_input = 'EDR alert: Detected exploitation attempt on web server. '

inputs = {'original_alert': user_input}
for event in app.stream(inputs, stream_mode='updates'):
    for node_name, node_state in event.items():
        print(f'\nNODO: [{node_name}]')

        if 'classification' in node_state:
            print(f"Classification: {node_state['classification']}")

        if 'mitre_data' in node_state:
            print(f"MITRE Data: {node_state['mitre_data'][:200]}...")

        if 'cve_data' in node_state:
            print(f"CVE Data: {node_state['cve_data'][:200]}...")

        if 'final_report' in node_state:
            print(f"Final Report: {node_state['final_report']}")


NODO: [classification]
Classification: mitre_search=True mitre_keywords='exploitation attempt, web server, web application exploitation, web server vulnerability exploitation' mitre_description='EDR alert indicates an exploitation attempt targeting a web server, suggesting malicious activity aimed at leveraging a vulnerability in the web server or its hosted web application.' cve_search=False cve_keywords='' cve_description=''


In [13]:
tracker.completion_tokens
tracker.total_tokens
tracker.node_metrics

{'classification': {'duration_s': 6.28,
  'prompt_tokens': 466,
  'completion_tokens': 312,
  'total_tokens': 778}}

# Pruebas búsqueda vectorial

In [156]:
# Crear dataset de descripciones y keywords (primer nodo)
import pandas as pd
import json

workflow = StateGraph(AgentState)
workflow.add_node('classification', classification_node)
workflow.set_entry_point('classification')
workflow.add_edge('classification', END)
app = workflow.compile()

df = pd.read_csv('data/unique_alerts.csv')
data = []

for i, row in df.iterrows():
    alert_data = json.loads(row['alert'])
    key_path = ['rule', 'mitre']  
    parent = alert_data
    for key in key_path[:-1]:
        parent = parent.get(key, {})
        if not isinstance(parent, dict):
            parent = {}
            break
    parent.pop(key_path[-1], None)
    labels = eval(row['real_ttps'])
    print(alert_data)
    print(labels)

    try:
        final_state = app.invoke({'original_alert': alert_data})
    except: 
        time.sleep(60)
        final_state = app.invoke({'original_alert': alert_data})

    data.append({
            'original_alert': alert_data,
            'description': final_state['classification'].mitre_description,
            'keywords': final_state['classification'].mitre_keywords,
            'alert_desc': alert_data['rule']['description'],
            'labels': labels
        }
    )

with open('data/unique_alerts_classification.json', 'w', encoding='utf-8') as file:
    json.dump(data, file, indent=4, ensure_ascii=False)


{'timestamp': '2025-09-22T18:37:00.601+0000', 'rule': {'level': 6, 'description': 'Processes running for all users were queried with ps command.', 'id': '92604', 'firedtimes': 1, 'mail': False, 'groups': ['audit_detections']}, 'agent': {'id': '002', 'name': 'videoserver', 'ip': '172.17.100.121'}, 'manager': {'name': 'wazuh'}, 'id': '1758566220.11498102', 'full_log': 'type=SYSCALL msg=audit(1758566218.934:4968): arch=c000003e syscall=59 success=yes exit=0 a0=55de6e506188 a1=55de6e4f6c90 a2=55de6e5060f8 a3=1b6 items=3 ppid=3135 pid=3136 auid=4294967295 uid=33 gid=33 euid=33 suid=33 fsuid=33 egid=33 sgid=33 fsgid=33 tty=pts0 ses=4294967295 comm="ps" exe="/usr/bin/ps" subj==unconfined key="T1166_Seuid_and_Setgid"\x1dARCH=x86_64 SYSCALL=execve AUID="unset" UID="www-data" GID="www-data" EUID="www-data" SUID="www-data" FSUID="www-data" EGID="www-data" SGID="www-data" FSGID="www-data" type=EXECVE msg=audit(1758566218.934:4968): argc=2 a0="ps" a1="auxwww" type=PATH msg=audit(1758566218.934:4968

In [ ]:
import pandas as pd
import json
import time

df = pd.read_csv('data/unique_alerts.csv')

metrics = {
    'alert': 0,
    'desc': 0,
    'keywords': 0
}
total_t = 0

for i, row in df.iterrows():
    alert_data = json.loads(row['alert'])
    key_path = ['rule', 'mitre']  
    parent = alert_data
    for key in key_path[:-1]:
        parent = parent.get(key, {})
        if not isinstance(parent, dict):
            parent = {}
            break
    parent.pop(key_path[-1], None)
    labels = eval(row['real_ttps'])
    print(alert_data)
    print(labels)

    try:
        final_state = app.invoke({'original_alert': alert_data})
    except: 
        time.sleep(60)
        final_state = app.invoke({'original_alert': alert_data})

    results1 = [final_state['mitre_data1'][i]['technique_id'] for i in range(config.mitre_top_k)]
    results2 = [final_state['mitre_data2'][i]['technique_id'] for i in range(config.mitre_top_k)]
    results3 = [final_state['mitre_data3'][i]['technique_id'] for i in range(config.mitre_top_k)]

    print(final_state['classification'].mitre_description)
    print(results1)

    print(final_state['classification'].mitre_keywords)
    print(results2)

    print(final_state['original_alert']['rule']['description'])
    print(results3)

    for t in labels:
        total_t += 1
        if t in results1:
            metrics['desc'] += 1
        if t in results2:
            metrics['keywords'] += 1
        if t in results3:
            metrics['alert'] += 1
    
    print(total_t)
    print(metrics)
    

{'timestamp': '2025-09-22T18:37:00.601+0000', 'rule': {'level': 6, 'description': 'Processes running for all users were queried with ps command.', 'id': '92604', 'firedtimes': 1, 'mail': False, 'groups': ['audit_detections']}, 'agent': {'id': '002', 'name': 'videoserver', 'ip': '172.17.100.121'}, 'manager': {'name': 'wazuh'}, 'id': '1758566220.11498102', 'full_log': 'type=SYSCALL msg=audit(1758566218.934:4968): arch=c000003e syscall=59 success=yes exit=0 a0=55de6e506188 a1=55de6e4f6c90 a2=55de6e5060f8 a3=1b6 items=3 ppid=3135 pid=3136 auid=4294967295 uid=33 gid=33 euid=33 suid=33 fsuid=33 egid=33 sgid=33 fsgid=33 tty=pts0 ses=4294967295 comm="ps" exe="/usr/bin/ps" subj==unconfined key="T1166_Seuid_and_Setgid"\x1dARCH=x86_64 SYSCALL=execve AUID="unset" UID="www-data" GID="www-data" EUID="www-data" SUID="www-data" FSUID="www-data" EGID="www-data" SGID="www-data" FSGID="www-data" type=EXECVE msg=audit(1758566218.934:4968): argc=2 a0="ps" a1="auxwww" type=PATH msg=audit(1758566218.934:4968

In [7]:
query1 = final_state['classification'].mitre_description
query2 = final_state['classification'].mitre_keywords
query3 = final_state['original_alert']['rule']['description']

query_vector1 = embedder.embed_query(query1)
query_vector2 = embedder.embed_query(query2)
query_vector3 = embedder.embed_query(query3)

[r['technique_id'] for r in mitre_store_v2.search_mitre_hybrid(query_vector=query_vector1, query_text=query2)]

NameError: name 'final_state' is not defined

In [10]:
query_vector = embedder.embed_query("The alert shows multiple HTTP GET requests containing directory traversal sequences (e.g., ../, %2e%2e/) targeting sensitive system files such as /etc/passwd, boot.ini, and winnt/repair/sam._, indicating an attempt to access unauthorized files on the web server.")
query_text = 'Path Traversal, Directory Traversal, Web Application Attack'

[r['technique_id'] for r in mitre_store_v2.search_mitre_hybrid(query_vector=query_vector, query_text=query_text)]

['T1599.001', 'T1056.003', 'T1689', 'T1204', 'T1189']

In [11]:
[r['technique_id'] for r in mitre_store_v2.search(query_vector)]

['T1204', 'T1204.001', 'T1189', 'T1552', 'T1056.003']

In [31]:
results1 = [final_state['mitre_data1'][i]['technique_id'] for i in range(config.mitre_top_k)]
results2 = [final_state['mitre_data2'][i]['technique_id'] for i in range(config.mitre_top_k)]
results3 = [final_state['mitre_data3'][i]['technique_id'] for i in range(config.mitre_top_k)]

print(final_state['classification'].mitre_description)
print(results1)

print(final_state['classification'].mitre_keywords)
print(results2)

print(final_state['original_alert']['rule']['description'])
print(results3)

The network interface ens3 on host inetfw has entered promiscuous mode, which allows the device to capture all network traffic passing through the interface. This behavior is often associated with network reconnaissance or traffic interception.
['T1016.001', 'T1011', 'T1090.001', 'T1040', 'T1041', 'T1018', 'T1029', 'T1071.001', 'T1020', 'T1016']
network sniffing, promiscuous mode, packet capture
['T1040', 'T1071.002', 'T1016.001', 'T1095', 'T1071.001', 'T1205.002', 'T1071', 'T1056', 'T1048', 'T1020']
Interface entered in promiscuous(sniffing) mode.
['T1056', 'T1041', 'T1040', 'T1205.002', 'T1011', 'T1016.001', 'T1205', 'T1001', 'T1546.005', 'T1095']


In [66]:
query = embedder.embed_query('which is vulnerable to a DLL side-loading vulnerability, and loads a file')

results = mitre_store.search(query, top_k=10)

[results[i]['technique_id'] for i in range(10)]

['T1574.001',
 'T1055.001',
 'T1546.010',
 'T1620',
 'T1129',
 'T1218.011',
 'T1574.008',
 'T1547.002',
 'T1574.014',
 'T1574.004']

In [ ]:
from mitreattack.stix20 import MitreAttackData

mitre_data = MitreAttackData('data/enterprise-attack.json')
techniques = mitre_data.get_techniques(remove_revoked_deprecated=False) 

False


In [74]:
deprecated = []
for t in techniques:
    if t.get("revoked") or t.get("x_mitre_deprecated"):
        print(t.get("revoked"))
        print(t.get("x_mitre_deprecated"))
        print(mitre_data.get_attack_id(t.id))
        deprecated.append(mitre_data.get_attack_id(t.id))

True
False
T1066
True
False
T1156
True
False
T1067
True
False
T1143
True
False
T1161
True
False
T1150
True
False
T1148
True
False
T1492
True
False
T1044
True
False
T1171
True
False
T1501
True
False
T1514
True
False
T1109
True
False
T1099
True
False
T1163
True
False
T1116
True
False
T1522
True
False
T1093
True
False
T1172
True
False
T1178
True
False
T1013
True
False
T1192
True
False
T1121
True
False
T1206
True
False
T1063
True
False
T1167
True
False
T1527
True
False
T1562.009
True
False
T1180
True
False
T1165
True
False
T1070.002
True
False
T1089
True
False
T1487
True
False
T1214
True
False
T1103
True
False
T1017
True
False
T1162
True
False
T1058
True
False
T1024
True
False
T1536
True
False
T1562
True
False
T1079
True
False
T1139
True
False
T1503
False
True
T1153
True
False
T1038
True
False
T1050
True
False
T1032
False
True
T1062
True
False
T1182
True
False
T1562.002
True
False
T1004
True
False
T1009
True
False
T1076
True
False
T1131
True
False
T1181
True
False
T1562.004
True
False
T115

In [78]:
with open('data/single_label.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

data

[{'text': 'This file extracts credentials from LSASS similar to Mimikatz.',
  'label': 'T1003.001',
  'doc_title': 'NotPetya Technical Analysis  A Triple Threat File Encryption MFT Encryption Credential Theft'},
 {'text': 'It calls OpenProcess on lsass.exe with access flag set to VM_READ, and looks for the modules wdigest.dll and lsasrv.dll loaded in the lsass.exe process.',
  'label': 'T1003.001',
  'doc_title': 'NotPetya Technical Analysis  A Triple Threat File Encryption MFT Encryption Credential Theft'},
 {'text': 'It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.',
  'label': 'T1210',
  'doc_title': 'NotPetya Technical Analysis  A Triple Threat File Encryption MFT Encryption Credential Theft'},
 {'text': 'SMB exploitation via EternalBlue',
  'label': 'T1210',
  'doc_title': 'NotPetya Technical Analysis  A Triple Threat File Encryption MFT Encryption Credential Theft'

In [93]:
dep_count = 0
dep_ex = []


for e in data:
    _, label, _ = e.values()
    if label in deprecated:
        dep_count += 1
        dep_ex.append(label)

print(dep_count)
set(dep_ex)

176


{'T1562.001', 'T1574.002'}

In [92]:
mapping_dict = {
    'T1562.001': 'T1685',
    'T1574.002': 'T1574.001'
}

new_data = []

for e in data:
    t, label, _ = e.values()
    new_data.append({'text': t,
                        'label': mapping_dict.get(label, label)})
        
with open('data/tram_data.json', 'w', encoding='utf-8') as file:
    json.dump(new_data, file, indent=4, ensure_ascii=False)

In [145]:
with open('data/tram_data.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

data

[{'text': 'This file extracts credentials from LSASS similar to Mimikatz.',
  'label': 'T1003.001'},
 {'text': 'It calls OpenProcess on lsass.exe with access flag set to VM_READ, and looks for the modules wdigest.dll and lsasrv.dll loaded in the lsass.exe process.',
  'label': 'T1003.001'},
 {'text': 'It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.',
  'label': 'T1210'},
 {'text': 'SMB exploitation via EternalBlue', 'label': 'T1210'},
 {'text': 'SMBv1 Exploitation via EternalBlue', 'label': 'T1210'},
 {'text': 'has the capability to exploit SMBv1 via the well known EternalBlue exploit.',
  'label': 'T1210'},
 {'text': 'SMB copy and remote execution', 'label': 'T1570'},
 {'text': 'This thread is then used to execute the SMB copy and remote execution',
  'label': 'T1570'},
 {'text': 'SMB copy and remote execution', 'label': 'T1570'},
 {'text': 'SMB Copy and Remote Executi

In [5]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9838.31it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:

text,label,_ = data[0].values()
# print(text, label)

query = embedder.embed_query(text)

results = mitre_store.search(query, top_k=50)

pares = []
for r in results:
    pares.append((text, r['description']))

scores = reranker.predict(pares)

for i,doc in enumerate(results):
    doc['rerank_score'] = float(scores[i])

reordenados = sorted(results, key=lambda x: x['rerank_score'], reverse=True)

top_10_final = reordenados[:10]

results_10_inicial = [results[i]['technique_id'] for i in range(10)]
results_10_final = [top_10_final[i]['technique_id'] for i in range(10)]

print(label)
print(results_10_inicial)
print(results_10_final)


T1003.001
['T1003.001', 'T1003.004', 'T1003.002', 'T1003', 'T1552', 'T1003.005', 'T1552.002', 'T1552.001', 'T1552.006', 'T1555.003']
['T1003.001', 'T1003.004', 'T1003', 'T1558.005', 'T1003.006', 'T1003.005', 'T1003.002', 'T1555.003', 'T1552', 'T1555.004']


In [146]:
from tqdm import tqdm

k_valores = [1,3,5,10,50,100]

aciertos = {k:0 for k in k_valores}
# aciertos2 = {k:0 for k in k_valores}

fallos = []
for e in tqdm(data, desc='Evaluando embeddings'):
    text,label = e.values()
    # print(text, label)

    query = emb.embed_query(text)

    results = mitre_store_v3_bge.search(query, top_k=100)
    # results = mitre_store_v2.search_mitre_hybrid(query, text, top_k=100)

    # pares = []
    # for r in results:
    #     pares.append((text, r['description']))

    # scores = reranker.predict(pares)

    # for i,doc in enumerate(results):
    #     doc['rerank_score'] = float(scores[i])

    # reordenados = sorted(results, key=lambda x: x['rerank_score'], reverse=True)

    # top_10_final = reordenados[:10]

    results_10 = [results[i]['technique_id'] for i in range(100)]
    # results_10_final = [reordenados[i]['technique_id'] for i in range(100)]

    
    if label in results_10:
        top = results_10.index(label)
        for k in k_valores:
            if top < k:
                aciertos[k] += 1

    if label not in results_10[:10]:
        if label in results_10:
            top = results_10.index(label)
        else:
            top = '+100'

        fallos.append((text, label, top))
    # if label in results_10_final:
    #     top = results_10_final.index(label)
    #     for k in k_valores:
    #         if top < k:
    #             aciertos2[k] += 1

    
accuracy = {f'Top-{k}': (hits / len(data)) for k, hits in aciertos.items()}
# accuracy2 = {f'Top-{k}': (hits / len(data[:100])) for k, hits in aciertos2.items()}

Evaluando embeddings: 100%|██████████| 5089/5089 [09:15<00:00,  9.17it/s]


In [147]:
accuracy

{'Top-1': 0.2749066614266064,
 'Top-3': 0.46394183533110633,
 'Top-5': 0.5588524268029083,
 'Top-10': 0.6763607781489487,
 'Top-50': 0.8693259972489684,
 'Top-100': 0.921202593829829}

In [39]:
fallos

[('SMB exploitation via EternalBlue', 'T1210', 14),
 ('SMBv1 Exploitation via EternalBlue', 'T1210', 22),
 ('SMB copy and remote execution', 'T1570', 57),
 ('This thread is then used to execute the SMB copy and remote execution',
  'T1570',
  '+100'),
 ('SMB copy and remote execution', 'T1570', 57),
 ('SMB Copy and Remote Execution', 'T1570', 57),
 ('and the same key is used to decrypt the MFT.', 'T1140', 17),
 ('its resource section are decompressed and written to disk', 'T1140', 15),
 ('Once the command line arguments are generated', 'T1059.003', 14),
 ('The malware also spawns cmd.exe to execute the following command',
  'T1059.003',
  27),
 ('Process Hashes and Process Privilege Checks', 'T1057', 14),
 ('compares each hash with 3 hardcoded hashes:\n\n0x6403527E → avp.exe associated with Kaspersky AV\n0x23214B44\xa0 → ns.exe associated with Norton Security\n0x651B3005 → ccSvcHst.exe associated with Symantec',
  'T1518.001',
  99),
 ('GetExtendedTcpTable to retrieve a list of TCP end

In [23]:
fallos

[('It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.',
  'T1210',
  19),
 ('This thread is then used to execute the SMB copy and remote execution',
  'T1570',
  21),
 ('The malware decompresses its resource named', 'T1140', 12),
 ('decrypt the MFT,', 'T1140', 10),
 ('Once the sector is decrypted,', 'T1140', 40),
 ('and the same key is used to decrypt the MFT.', 'T1140', 42),
 ('is also decoded, and placed back', 'T1140', 26),
 ('its resource section are decompressed and written to disk', 'T1140', 62),
 ('that hashes each running process on the system', 'T1057', 13),
 ('Process Hashes and Process Privilege Checks', 'T1057', 14),
 ('GetExtendedTcpTable to retrieve a list of TCP endpoints\nGetIpNetTable to retrieve',
  'T1106',
  '+100'),
 ('NetServerEnum to get a list', 'T1106', '+100'),
 ('NetServerGetInfoto retrieve the current configuration', 'T1106', '+100'),
 ('CreateF

In [18]:
accuracy

{'Top-1': 0.2532914128512478,
 'Top-3': 0.43033994890941246,
 'Top-5': 0.5103163686382394,
 'Top-10': 0.625466692866968,
 'Top-50': 0.8661819610925525,
 'Top-100': 0.9345647474945962}

In [ ]:
query = local_embedder.embed_query('uses HTTPS to communicate with its C2 servers')

results = mitre_store_bge.search(query, top_k=10)

[results[i]['technique_id'] for i in range(10)]

['T1505',
 'T1195.003',
 'T1114',
 'T1012',
 'T1136.002',
 'T1573.001',
 'T1132.001',
 'T1016.001',
 'T1011.001',
 'T1001.001']

In [ ]:
text = "The user www-data executed the command 'ps auxwww' to list all running processes on the system. This behavior is associated with process discovery and is flagged by an audit rule related to potential privilege escalation or suspicious activity involving setuid/setgid binaries."
vector = mitre_store_bge.search(local_embedder.embed_query(text))[0]['vector']

In [ ]:
r = mitre_store.search(embedder.embed_query(text))
[r[i]['technique_id'] for i in range(5)]

['T1134.004', 'T1036.011', 'T1003.007', 'T1543.004', 'T1057']

In [ ]:
r2 = mitre_store_bge.search(local_embedder.embed_query(text))
[r2[i]['technique_id'] for i in range(5)]

['T1205.001', 'T1127.001', 'T1119', 'T1587.002', 'T1602']

In [28]:
import numpy as np

def calcular_coseno(v1,v2):
    dot_product = np.dot(v1,v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)

    return dot_product / (norm1 * norm2)

t1 = 'Alert: Execution detected of a Powershell command coding in Base64'
t2 = 'Security: A Powershell script attempted to establish an external connection not authorized'
t3 = 'Add three cups of milk in a bowl and bake with hot butter'

e1 = embedder.embed_query(t1)
e2 = embedder.embed_query(t2)
e3 = embedder.embed_query(t3)

s1 = calcular_coseno(e1,e2)
s2 = calcular_coseno(e1, e3)

print(s1)
print(s2)

0.48960734562458924
-0.01450588246579226


In [139]:
emb = SentenceTransformerEmbedder('BAAI/bge-small-en-v1.5')
mitre_store_v3_bge = ElasticSearchVectorStore(index_name='mitre_attack_v3_bge_small')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5201.35it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [144]:
query = emb.embed_query("Valid Accounts, SSH, Remote Access")

results = mitre_store_v3_bge.search(query, top_k=10)

[results[i]['technique_id'] for i in range(10)]

['T1021.004',
 'T1098.004',
 'T1078',
 'T1021',
 'T1021.001',
 'T1133',
 'T1563.001',
 'T1552.001',
 'T1003.002',
 'T1078.001']